# terrasync — Explorador de Dados Geoespaciais

Navegue pelos parquets bronze/silver e visualize em mapa interativo Leaflet.

In [1]:
from pathlib import Path

import geopandas as gpd
import folium
from IPython.display import display, HTML

## Descobrir arquivos disponíveis

Lista todos os `.parquet` nas pastas `bronze/` e `silver/`.

In [2]:
DATA_DIR = Path("../data")

parquets = sorted(DATA_DIR.rglob("*.parquet"))
for p in parquets:
    print(p.relative_to(DATA_DIR))

bronze/ana/ana_pivos_irrigacao.parquet
bronze/deter/deter_cerrado.parquet
bronze/funai/funai_tis_poligonais.parquet
bronze/ibama/ibama_vw_brasil_adm_embargo_a.parquet
bronze/icmbio/icmbio_limiteucsfederais_a.parquet
bronze/incra/incra_quilombolas_lim_quilombolas_a.parquet
bronze/prodes/prodes_caatinga_yearly_deforestation.parquet
bronze/prodes/prodes_cerrado_yearly_deforestation.parquet
bronze/prodes/prodes_mata_atlantica_yearly_deforestation.parquet
bronze/prodes/prodes_pampa_yearly_deforestation.parquet
bronze/prodes/prodes_pantanal_yearly_deforestation.parquet
bronze/sicar/sicar_ac.parquet
bronze/sicar/sicar_al.parquet
bronze/sicar/sicar_am.parquet
bronze/sicar/sicar_ap.parquet
bronze/sicar/sicar_ba.parquet
bronze/sicar/sicar_ce.parquet
bronze/sicar/sicar_df.parquet
bronze/sicar/sicar_es.parquet
bronze/sicar/sicar_go.parquet
bronze/sicar/sicar_ma.parquet
bronze/sicar/sicar_mg.parquet
bronze/sicar/sicar_ms.parquet
bronze/sicar/sicar_mt.parquet
bronze/sicar/sicar_pa.parquet
bronze/sic

## Selecionar e carregar um parquet

Altere o `ARQUIVO` abaixo para qualquer path listado acima.

In [3]:
ARQUIVO = "bronze/prodes/prodes_mata_atlantica_yearly_deforestation.parquet"  # ← altere aqui

gdf = gpd.read_parquet(DATA_DIR / ARQUIVO)
print(f"Features: {len(gdf):,}  |  Colunas: {list(gdf.columns)}")

Features: 1,102,761  |  Colunas: ['geometry', 'uid', 'state', 'path_row', 'main_class', 'class_name', 'def_cloud', 'julian_day', 'image_date', 'year', 'area_km', 'scene_id', 'source', 'satellite', 'sensor', 'uuid', 'publish_year']


## Mapa interativo

Mapa Leaflet com popup dos atributos ao clicar. Para datasets grandes, amostra `LIMIT` features.

In [4]:
LIMIT = 3000  # máximo de features no mapa (0 = sem limite)

sample = gdf if (LIMIT == 0 or len(gdf) <= LIMIT) else gdf.sample(n=LIMIT, random_state=42)

# Reprojetar para WGS84 se necessário
if sample.crs and sample.crs.to_epsg() != 4326:
    sample = sample.to_crs(epsg=4326)

# Centróide do bounding box
bounds = sample.total_bounds  # [minx, miny, maxx, maxy]
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

m = folium.Map(location=center, zoom_start=5, tiles="OpenStreetMap")

# Tooltip com todos os atributos (exceto geometry)
tooltip_fields = [c for c in sample.columns if c != "geometry"]
tooltip_aliases = tooltip_fields

folium.GeoJson(
    sample,
    name=Path(ARQUIVO).stem,
    style_function=lambda _: {
        "color": "#e63946",
        "weight": 1.5,
        "fillOpacity": 0.25,
    },
    tooltip=folium.GeoJsonTooltip(fields=tooltip_fields[:8], aliases=tooltip_aliases[:8]),
    popup=folium.GeoJsonPopup(fields=tooltip_fields, aliases=tooltip_aliases),
).add_to(m)

folium.LayerControl().add_to(m)

# Fit bounds
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

m